# GVAF — 산업별 부가가치 VARX 파일럿 🧪

**목적(교수님 지시):** 국민계정에서 직접 나오는 **~30개 산업**의 부가가치로,
외생변수(**유가·환율·금리·취업자수**, 가능하면 **세계교역량**)를 넣은 **VARX**를
돌려서 **"가능성만"** 본다. (정확도가 아니라 파이프라인이 끝까지 도는지 확인)

**흐름:** ① 데이터 → ② 정상성 검정·차분 → ③ VARX 추정 → ④ 진단 → ⑤ 예측

> 기본은 **데모(합성) 데이터**라 코랩에서 바로 `런타임 → 모두 실행`하면 끝까지 돕니다.
> 실데이터는 `CONFIG['DATA_MODE']='csv'`로 바꾸고 KOSIS/ECOS에서 받은 CSV를 업로드하세요.

In [ ]:
# Colab 기본 패키지로 충분 (numpy, pandas, statsmodels, matplotlib)
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.diagnostic import acorr_ljungbox
np.set_printoptions(suppress=True, precision=3)
plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.grid'] = True
print('setup OK')

## ⚙️ 설정 (CONFIG)
- `DATA_MODE`: `'demo'`(합성, 즉시 실행) 또는 `'csv'`(실데이터 업로드)
- `N_INDUSTRIES`: 산업 수(데모용). 실데이터면 CSV 열 수가 자동 사용됨
- `LAGS`: VAR 시차. **연간·소표본이면 1 권장**
- `EXOG`: 외생변수 — 유가·환율·금리·취업자수·세계교역량
- `RIDGE`: `None`=자동(모수>표본이면 수축 자동 ON) / `0.0`=순수 OLS / `>0`=수동

In [ ]:
CONFIG = {
    'DATA_MODE': 'demo',          # 'demo' | 'csv'
    'N_INDUSTRIES': 30,
    'LAGS': 1,
    'EXOG': ['oil', 'fx', 'rate', 'emp', 'trade'],   # 유가 환율 금리 취업자수 세계교역량
    'FORECAST_STEPS': 8,
    'RIDGE': None,                # None=자동 / 0.0=OLS / >0=수동 수축
    'SEED': 7,
}
CONFIG

## ① 데이터 준비

**데모**: ~30개 산업의 로그 부가가치를 *공통 거시요인 + 외생 약효과 + 고유 확률보행(I(1))* 으로 합성.
외생변수도 현실처럼 대부분 I(1)로 생성(환율·금리는 수준이 떠도는 성질).

**실데이터(참고 출처)**
- 산업별 부가가치: **국민계정 경제활동별 GDP(명목/실질)** — KOSIS/ECOS. "제일 하위 + 겹치지 않는" 분류로 받으면 ~30개.
- 외생: 환율·금리·유가 → **ECOS** / 취업자수 → **경제활동인구조사(KOSIS)** / 세계교역량 → **IMF·CPB World Trade Monitor**.
- CSV 형식: 1열은 날짜(index), 나머지 열은 각 시계열. (산업 CSV 1개 + 외생 CSV 1개)

In [ ]:
def make_demo(n_ind, T=120, seed=7):
    rng = np.random.default_rng(seed)
    idx = pd.period_range('1995Q1', periods=T, freq='Q').to_timestamp()
    f = np.cumsum(rng.normal(0.004, 0.02, T))                 # 공통 거시요인 (I(1))
    oil   = 4.0 + np.cumsum(rng.normal(0, 0.06, T))            # 로그 유가
    fx    = 7.0 + np.cumsum(rng.normal(0, 0.02, T))            # 로그 USD/KRW (떠돎)
    rate  = 2.5 + np.cumsum(rng.normal(0, 0.05, T))            # 금리 (지속성↑)
    emp   = 16.0 + np.cumsum(rng.normal(0.002, 0.01, T))       # 로그 취업자수
    trade = 5.0 + 0.8*f + np.cumsum(rng.normal(0, 0.04, T))    # 세계교역량 (요인과 상관)
    X = pd.DataFrame({'oil':oil, 'fx':fx, 'rate':rate, 'emp':emp, 'trade':trade}, index=idx)
    load = rng.uniform(0.3, 1.2, n_ind)
    bx   = rng.normal(0, 0.25, (n_ind, 2))                     # oil, trade 약효과
    Y = np.zeros((T, n_ind))
    for i in range(n_ind):
        shock = rng.normal(0.003, 0.03, T)
        Y[:, i] = (9 + load[i]*f
                   + bx[i,0]*(oil-oil.mean()) + bx[i,1]*(trade-trade.mean())
                   + np.cumsum(shock))                          # 로그 부가가치 수준 (I(1))
    Y = pd.DataFrame(Y, index=idx, columns=[f'ind{i+1:02d}' for i in range(n_ind)])
    return Y, X

def load_csv():
    from google.colab import files
    print('① 산업별 부가가치 CSV, ② 외생변수 CSV (oil,fx,rate,emp,trade) 순서로 업로드')
    up = files.upload(); names = list(up.keys())
    Y = pd.read_csv(names[0], index_col=0, parse_dates=True)
    X = pd.read_csv(names[1], index_col=0, parse_dates=True)
    return Y, X

if CONFIG['DATA_MODE'] == 'demo':
    Y_lvl, X_lvl = make_demo(CONFIG['N_INDUSTRIES'], seed=CONFIG['SEED'])
else:
    Y_lvl, X_lvl = load_csv()

print('내생(산업):', Y_lvl.shape, ' | 외생:', X_lvl.shape, list(X_lvl.columns))
Y_lvl.iloc[:3, :5]

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
Y_lvl.iloc[:, :6].plot(ax=ax[0], legend=False, title='산업 부가가치(로그수준, 일부)')
X_lvl.plot(ax=ax[1], title='외생변수(수준)')
plt.tight_layout(); plt.show()

## ② 정상성 검정 & 차분

VARX는 **정상(stationary)** 시계열을 가정한다. 수준(level)이 *확률적 추세(단위근)* 를 가지면
(목줄 풀린 강아지처럼 떠돔) **차분**해서 넣어야 한다. 안 그러면 **가짜 회귀(spurious regression)**.

- 산업 부가가치(로그수준) → **로그차분 = 성장률** 로 변환
- 외생변수 → ADF로 검정해 **I(0)이면 수준, 아니면 차분**
- ADF p<0.05 → 정상(I(0)) 판정

In [ ]:
def adf_p(s):
    s = pd.Series(s).dropna()
    if len(s) < 8: return np.nan
    try: return adfuller(s, autolag='AIC')[1]
    except Exception: return np.nan

def stationarity_table(df, label):
    rows = []
    for c in df.columns:
        p0, p1 = adf_p(df[c]), adf_p(df[c].diff())
        order = 'I(0)' if (p0 is not None and p0 < 0.05) else ('I(1)' if (p1 is not None and p1 < 0.05) else 'I(2)+?')
        rows.append((c, round(p0,3) if p0==p0 else None, round(p1,3) if p1==p1 else None, order))
    return pd.DataFrame(rows, columns=[label, 'ADF p(수준)', 'ADF p(차분)', '적분차수'])

print('=== 외생변수 정상성 ==='); display(stationarity_table(X_lvl, 'exog'))

# 변환
dY = Y_lvl.diff().dropna()                      # 로그수준 차분 = 성장률 (산업은 일괄 1차차분)
Xs = pd.DataFrame(index=X_lvl.index)
for c in X_lvl.columns:
    p0 = adf_p(X_lvl[c])
    Xs[c] = X_lvl[c] if (p0 is not None and p0 < 0.05) else X_lvl[c].diff()
Xs = Xs.dropna()
common = dY.index.intersection(Xs.index)
dY, Xs = dY.loc[common], Xs.loc[common]
print('정상화 후 — 내생(성장률):', dY.shape, ' 외생:', Xs.shape)

## ③ VARX 추정

각 산업 성장률을 *[상수 + 모든 산업의 과거 성장률(시차 p) + 동시기 외생변수]* 에 회귀.

**핵심:** 산업이 N개면 방정식당 회귀계수가 `1 + N·p + k`개 → **N이 커지면 모수 폭증(차원의 저주).**
표본보다 모수가 많아지면 OLS가 불안정 → **ridge(수축)** 자동 적용. (이게 "Large VAR은 수축 필수"의 실연)

In [ ]:
def fit_varx(Yarr, Xarr, p=1, ridge=None):
    T, n = Yarr.shape; k = Xarr.shape[1]
    Z, tgt = [], []
    for t in range(p, T):
        row = [1.0]
        for l in range(1, p+1): row += list(Yarr[t-l])
        row += list(Xarr[t])                 # 동시기 외생
        Z.append(row); tgt.append(Yarr[t])
    Z, Yt = np.array(Z), np.array(tgt)
    m, neff = Z.shape[1], Z.shape[0]
    auto = (ridge is None and m >= neff)     # 과대모수면 수축 자동
    lam = 1.0 if auto else (0.0 if ridge is None else float(ridge))
    B = np.linalg.solve(Z.T@Z + lam*np.eye(m), Z.T@Yt)
    resid = Yt - Z@B
    return dict(B=B, Z=Z, Yt=Yt, resid=resid, p=p, n=n, k=k, m=m, neff=neff, lam=lam, auto=auto)

p = CONFIG['LAGS']
fit = fit_varx(dY.values, Xs.values, p=p, ridge=CONFIG['RIDGE'])
print('표본 %d개 × 방정식당 계수 %d개  (내생 %d, 외생 %d, 시차 %d)' % (fit['neff'], fit['m'], fit['n'], fit['k'], p))
print('ridge λ = %.2f  %s' % (fit['lam'], '← 자동 수축 ON (모수>=표본)' if fit['auto'] else ''))
ss_res = (fit['resid']**2).sum(0); ss_tot = ((fit['Yt']-fit['Yt'].mean(0))**2).sum(0)
r2 = 1 - ss_res/np.where(ss_tot==0, np.nan, ss_tot)
print('산업별 R² — 중앙값 %.2f (min %.2f, max %.2f)' % (np.nanmedian(r2), np.nanmin(r2), np.nanmax(r2)))

## ④ 진단
- **안정성**: 동반행렬 최대 |고유값| < 1 이면 안정(예측이 발산 안 함)
- **잔차 자기상관**: Ljung-Box p>0.05 면 백색잡음(설명 잘 됨)

In [ ]:
n, p, B = fit['n'], fit['p'], fit['B']
A_list, off = [], 1
for l in range(p):
    A_list.append(B[off:off+n, :].T); off += n     # lag l+1 계수행렬 (n,n)
if p == 1:
    comp = A_list[0]
else:
    top = np.hstack(A_list)
    comp = np.vstack([top, np.hstack([np.eye(n*(p-1)), np.zeros((n*(p-1), n))])])
eig = np.abs(np.linalg.eigvals(comp))
print('안정성: 최대 |고유값| = %.3f → %s' % (eig.max(), '안정 ✅' if eig.max() < 1 else '불안정 ⚠️'))

print('잔차 백색잡음 검정(대표 산업):')
for j in range(min(3, n)):
    try:
        pval = acorr_ljungbox(fit['resid'][:, j], lags=[4])['lb_pvalue'].iloc[0]
        print('  산업%d  Ljung-Box(4) p=%.3f  %s' % (j+1, pval, '백색잡음 OK' if pval > 0.05 else '자기상관 잔존'))
    except Exception as e:
        print('  산업%d  검정 skip (%s)' % (j+1, e))

## ⑤ 시나리오 예측 (가능성 확인)

외생변수의 미래 경로를 주면 산업별 성장률 → 부가가치 수준을 예측한다.
데모는 *최근 평균 변화*를 미래에 유지(실제론 시나리오 S1~S4별 외생경로로 교체).

In [ ]:
h = CONFIG['FORECAST_STEPS']
X_future = np.tile(Xs.values[-12:].mean(0), (h, 1))     # 데모용 시나리오(최근 평균 유지)

def forecast(fit, Y_hist_diff, X_future):
    p, B = fit['p'], fit['B']
    hist = list(Y_hist_diff[-p:]); out = []
    for t in range(len(X_future)):
        row = [1.0]
        for l in range(1, p+1): row += list(hist[-l])
        row += list(X_future[t])
        yhat = np.array(row) @ B
        out.append(yhat); hist.append(yhat)
    return np.array(out)

fc_diff = forecast(fit, dY.values, X_future)            # 성장률(차분) 예측
fc_lvl = Y_lvl.iloc[-1].values + np.cumsum(fc_diff, axis=0)   # 로그수준 복원

xh = np.arange(len(Y_lvl)); xf = np.arange(len(Y_lvl), len(Y_lvl)+h)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for j in range(min(4, fit['n'])):
    line, = ax[0].plot(xh, Y_lvl.iloc[:, j].values, lw=1)
    ax[0].plot(xf, fc_lvl[:, j], '--', color=line.get_color())
ax[0].set_title('산업 부가가치(로그수준) + 예측(점선)')
ax[1].bar(range(fit['n']), fc_diff.mean(0))
ax[1].set_title('산업별 평균 예측 성장률')
plt.tight_layout(); plt.show()
print('✅ 파이프라인이 끝까지 돌았습니다 — 가능성 확인 완료')

## ✅ 결론 & 다음 단계

이 노트북은 **"돌아가는지(feasibility)"** 를 보는 파일럿입니다. 끝까지 돌면 통과.

**현실 점검 (중요)**
- 연간 국민계정(약 35개 시점)에 산업 30개 → 방정식당 계수 ~36개 → **모수 ≈ 표본** = OLS 불안정.
  - 위 코드는 이때 **ridge 자동 ON**으로 돌아가지만, 본 모형은 **Large Bayesian VAR**(Minnesota + 산업연관표 prior, Bańbura et al. 2010)로 가야 함.
  - 또는 **분기 자료**(2000Q1~, ~100개 시점)를 쓰면 표본이 늘어 더 안정적.
- 취업자수를 외생으로 둔 건 파일럿 가정(원래 노동은 내생일 수 있음).

**실데이터로 돌리기**
1. KOSIS 국민계정 경제활동별 부가가치 → 산업 CSV (겹치지 않는 최하위 분류)
2. ECOS(환율·금리·유가) + 경활조사(취업자수) + IMF/CPB(세계교역량) → 외생 CSV
3. `CONFIG['DATA_MODE']='csv'` 로 바꾸고 모두 실행

**확장 경로:** VARX(파일럿) → **Large Bayesian VARX**(70개 산업, 수축) → 시나리오 팬차트(제안서 Block 2).